##### Task 5
Validate item references inside orders.

What I want in the end:
- A flattened view of order lines from the nested items array
- A table of order lines whose item_id does not exist in items
- Count of invalid item references
- List of affected order_id and item_id pairs

In [0]:
%sql
USE SCHEMA e_commerce;

In [0]:
%sql
-- A flattened view of order lines from the nested items array

SELECT
  order_id,
  customer_id,
  item.item_id AS item_id,
  item.quantity AS quantity
FROM bronze_orders
LATERAL VIEW explode(items) AS item
ORDER BY order_id

In [0]:
%sql
-- A table of order lines whose item_id does not exist in items

SELECT o.order_id, o.customer_id, o.item_id, o.quantity
FROM (
  SELECT order_id, customer_id, item.item_id, item.quantity
  FROM bronze_orders
  LATERAL VIEW explode(items) AS item
) AS o
LEFT JOIN bronze_items AS i
  ON o.item_id = i.item_id
WHERE i.item_id IS NULL;

In [0]:
%sql
-- Count of invalid item references

SELECT COUNT(*) invalid_item_ref
FROM (
  SELECT o.order_id, o.item_id
  FROM (
    SELECT order_id, item.item_id
    FROM bronze_orders
    LATERAL VIEW explode(items) AS item
  ) o
  LEFT JOIN bronze_items i
    ON o.item_id = i.item_id
  WHERE i.item_id IS NULL
) t

In [0]:
%sql
-- List of affected order_id and item_id pairs

SELECT o.order_id, o.item_id
FROM (
  SELECT order_id, item.item_id
  FROM bronze_orders
  LATERAL VIEW explode(items) AS item
) o
LEFT JOIN bronze_items i
  ON o.item_id = i.item_id
WHERE i.item_id IS NULL;

